# HyperTable Showcase

A Hypergraph graph where **each node output is a stored column** in a LanceDB table.

This notebook walks through every feature built in v1:
1. Define nodes and components
2. Construct a HyperTable
3. Insert with automatic derivation
4. Read back rows
5. Grain boundaries via `map_over` (parent → child tables)
6. LanceDB vector search
7. Per-column provenance and incrementality metadata
8. Component swap

## 1. Define Nodes and Components

Nodes are regular Python functions decorated with `@node`. Each produces one named output → one stored column.

Components (like `Embedder`) are injected via `.bind()` — they participate in derivation but are **not** stored as columns.

In [ ]:
import tempfile
from typing import TypedDict

from hypergraph import Graph, node
from hypergraph.materialization import HyperTable
from hypergraph.runners import SyncRunner


class Embedder:
    """A simple embedder component. In production, wraps OpenAI/Cohere/etc."""
    def __init__(self, model_name: str = "test-embed", dim: int = 8):
        self.model_name = model_name
        self.dim = dim

    def _config(self):
        return {"model": self.model_name, "dim": self.dim}

    def embed(self, text: str) -> list[float]:
        vec = [0.0] * self.dim
        for i, c in enumerate(text[:self.dim]):
            vec[i] = float(ord(c)) / 122.0
        return vec


@node(output_name="clean_text")
def clean(text: str) -> str:
    return text.strip().lower()


@node(output_name="vector")
def embed_text(clean_text: str, embedder: Embedder) -> list[float]:
    return embedder.embed(clean_text)


@node(output_name="word_count")
def count_words(clean_text: str) -> int:
    return len(clean_text.split())


STORE_DIR = tempfile.mkdtemp(prefix="hypertable_demo_")
print(f"Store directory: {STORE_DIR}")

## 2. Construct a HyperTable

Pass your nodes, an `identity` column, and a LanceDB `store` URI.

Chain `.bind()` for components and `.with_runner()` for the execution engine.

`vector_columns={"vector": 8}` tells the schema builder to use a fixed-size list — required for LanceDB ANN search.

In [ ]:
embedder = Embedder(dim=8)

docs = HyperTable(
    [clean, embed_text, count_words],
    identity="doc_id",
    store=f"lancedb://{STORE_DIR}/docs",
    vector_columns={"vector": 8},
).bind(embedder=embedder).with_runner(SyncRunner())

print("HyperTable 'docs' ready")
print(f"  identity:   doc_id")
print(f"  source:     text")
print(f"  derived:    clean_text, vector, word_count")
print(f"  bound:      embedder (not stored)")
print(f"  runner:     SyncRunner")

## 3. Insert and Derive

Insert via kwargs or a list of dicts. The graph runs automatically:
- Source columns (`text`) are stored as-is
- Derived columns (`clean_text`, `vector`, `word_count`) are computed by running the graph
- Internal metadata (provenance, fingerprint, write generation) is tracked

In [ ]:
# Batch insert
docs.insert([
    dict(doc_id="d1", text="  Hello World  "),
    dict(doc_id="d2", text="Goodbye Cruel World"),
    dict(doc_id="d3", text="Hello There Friend"),
])

# Single insert with extra metadata (schema evolves automatically)
docs.insert(doc_id="d4", text="The quick brown fox jumps", title="Fox Story")

print(f"Row count: {docs.count()}")

## 4. Read Back Rows

`.get(id)` returns a single row as a Python dict. Internal columns (`_provenance_*`, `_row_fingerprint`, `_write_gen`) are filtered out.

Numpy/Arrow types are normalized to plain Python (`list`, `int`, `float`).

In [ ]:
row = docs.get("d1")
print("Row d1:")
for k, v in row.items():
    display_v = f"{v[:3]}... ({len(v)}d)" if isinstance(v, list) else repr(v)
    print(f"  {k:15s} = {display_v}")

print(f"\ntype(vector): {type(row['vector']).__name__}")  # list, not numpy.ndarray

In [ ]:
# Metadata column 'title' was auto-added when d4 was inserted
row_d4 = docs.get("d4")
print(f"d4 title: {row_d4['title']}")
print(f"d4 word_count: {row_d4['word_count']}")

# Rows without 'title' have None
row_d1 = docs.get("d1")
print(f"d1 title: {row_d1.get('title')}")

## 5. LanceDB Vector Search

Since we declared `vector_columns={"vector": 8}`, the column uses a fixed-size list type.

This makes it searchable via LanceDB's native ANN index.

In [ ]:
import lancedb

db = lancedb.connect(f"{STORE_DIR}/docs")
lance_tbl = db.open_table("doc")

query_vec = embedder.embed("hello")
results = lance_tbl.search(query_vec, vector_column_name="vector").limit(3).to_pandas()

print("Vector search for 'hello' (top 3):")
for _, r in results.iterrows():
    print(f"  {r['doc_id']:5s}  {r['clean_text']:30s}  dist={r['_distance']:.4f}")

## 6. Grain Boundaries via `map_over`

When a node produces a list of items, `map_over` creates a **child table** — each item gets its own row, linked to the parent via `_parent_id`.

The child subgraph runs independently on each item.

In [ ]:
# --- Grain boundary nodes ---

@node(output_name="audio_path")
def extract_audio(path: str) -> str:
    return f"/tmp/{path.split('/')[-1]}.wav"

@node(output_name="transcript")
def transcribe(audio_path: str) -> str:
    return f"The quick brown fox jumps over {audio_path}"

class Utterance(TypedDict):
    utterance_id: str
    text: str
    speaker: str

@node(output_name="utterances")
def split_utterances(transcript: str) -> list[Utterance]:
    words = transcript.split()
    return [
        Utterance(utterance_id=f"u{i}", text=w, speaker="Alice" if i % 2 == 0 else "Bob")
        for i, w in enumerate(words)
    ]

# Subgraph for processing ONE utterance
process_utterance = Graph([clean, embed_text], name="process_utterance")

print("Grain boundary nodes defined")
print("  Root graph:  extract_audio → transcribe → split_utterances")
print("  Child graph: clean → embed_text (runs per utterance)")

In [ ]:
videos = HyperTable(
    [extract_audio, transcribe,
     split_utterances,
     process_utterance.as_node().map_over("utterances", identity="utterance_id")],
    identity="video_id",
    store=f"lancedb://{STORE_DIR}/videos",
    vector_columns={"vector": 8},
).bind(embedder=embedder).with_runner(SyncRunner())

videos.insert(video_id="v1", path="/data/meeting.mp4")
videos.insert(video_id="v2", path="/data/lecture.mp4")

print(f"Parent rows: {videos.count()}")
print(f"Child rows:  {videos.count('utterance')}")

In [ ]:
# Read parent row
parent = videos.get("v1")
print("Parent v1:")
print(f"  audio_path: {parent['audio_path']}")
print(f"  transcript: {parent['transcript'][:60]}...")

# Read children — each has identity, _parent_id, source (text), and derived columns
children = videos.children("v1")
print(f"\nChildren of v1: {len(children)} utterances")
print(f"Child columns: {list(children[0].keys())}")
for c in children[:5]:
    vec_preview = f"[{c['vector'][0]:.3f}, ...]" if c.get('vector') else 'N/A'
    print(f"  {c['utterance_id']:5s}  text={c['text']:10s}  clean={c['clean_text']:10s}  vec={vec_preview}")

## 7. Vector Search on Child Tables

Child tables are regular LanceDB tables — you can search them directly.

In [ ]:
db_videos = lancedb.connect(f"{STORE_DIR}/videos")
child_tbl = db_videos.open_table("utterance")

query_vec = embedder.embed("quick")
results = child_tbl.search(query_vec, vector_column_name="vector").limit(5).to_pandas()

print("Vector search on utterances for 'quick' (top 5):")
for _, r in results.iterrows():
    print(f"  {r['utterance_id']:5s}  text={r['text']:10s}  parent={r['_parent_id']}  dist={r['_distance']:.4f}")

## 8. Provenance and Incrementality Metadata

Every row tracks:
- `_row_fingerprint`: SHA-256 of all source values + node definitions + component configs
- `_provenance_{column}`: per-column content-key hash
- `_write_gen`: monotonic counter for crash-safe upserts

These enable incremental re-derivation: if the fingerprint matches, skip the row.

In [ ]:
# Peek at the raw LanceDB table to see internal columns
df = lance_tbl.to_pandas()
print("All columns in the docs table:")
for col in df.columns:
    sample = df[col].iloc[0]
    if isinstance(sample, (list, type(None))):
        print(f"  {col:30s}  {type(sample).__name__}")
    else:
        val = str(sample)[:50]
        print(f"  {col:30s}  {val}")

print(f"\nFingerprints differ across rows:")
for _, r in df.iterrows():
    print(f"  {r['doc_id']}: {r['_row_fingerprint'][:16]}...")

## 9. Component Swap

Swapping a component (e.g., a different embedder) changes the derived outputs. Each table is independent.

In [ ]:
# Small 4-dimensional embedder
emb_small = Embedder(model_name="small", dim=4)

docs_small = HyperTable(
    [clean, embed_text],
    identity="doc_id",
    store=f"lancedb://{STORE_DIR}/docs_small",
    vector_columns={"vector": 4},
).bind(embedder=emb_small).with_runner(SyncRunner())

docs_small.insert(doc_id="d1", text="Hello World")

row_8d = docs.get("d1")
row_4d = docs_small.get("d1")

print(f"8-dim embedder: vector length = {len(row_8d['vector'])}, values = {[round(v, 3) for v in row_8d['vector']]}")
print(f"4-dim embedder: vector length = {len(row_4d['vector'])}, values = {[round(v, 3) for v in row_4d['vector']]}")

## Summary

| Feature | How |
|---|---|
| **Construction** | `HyperTable([nodes], identity=, store=)` |
| **Components** | `.bind(embedder=...)` — used in derivation, not stored |
| **Runner** | `.with_runner(SyncRunner())` — separates execution from storage |
| **Insert** | `.insert(id=, text=)` or `.insert([{...}, {...}])` |
| **Read** | `.get(id)` → dict, `.count()`, `.children(parent_id)` |
| **Grain boundary** | `graph.as_node().map_over("items", identity="item_id")` |
| **Vector search** | `vector_columns={"vector": dim}` + LanceDB `.search()` |
| **Provenance** | `_row_fingerprint`, `_provenance_{col}`, `_write_gen` |
| **Schema evolution** | Extra kwargs auto-add metadata columns |